In [2]:
import os
import fastf1

In [3]:
os.makedirs(r"./../data/cache", exist_ok=True)
fastf1.Cache.enable_cache(r"./../data/cache")

In [4]:
session = fastf1.get_session(2024, "Monza", "R")
session.load()

core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '4', '55', '44', '1', '63', '11', '23', '20', '14', '43', '3', '31', '10', '77', '27', '24', '18', '22']


In [5]:
laps = session.laps
print(laps.columns)
laps[["Driver", "LapNumber", "LapTime", "Position", "Stint",
      "Compound", "TyreLife", "PitInTime", "PitOutTime",
      "TrackStatus", "Time"]].head()

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')


,Driver,LapNumber,LapTime,Position,Stint,Compound,TyreLife,PitInTime,PitOutTime,TrackStatus,Time
0,LEC,1.0,0 days 00:01:28.179000,2.0,1.0,MEDIUM,1.0,NaT,NaT,1,0 days 00:57:18.931000
1,LEC,2.0,0 days 00:01:25.396000,2.0,1.0,MEDIUM,2.0,NaT,NaT,1,0 days 00:58:44.327000
2,LEC,3.0,0 days 00:01:25.179000,2.0,1.0,MEDIUM,3.0,NaT,NaT,1,0 days 01:00:09.506000
3,LEC,4.0,0 days 00:01:24.810000,2.0,1.0,MEDIUM,4.0,NaT,NaT,1,0 days 01:01:34.316000
4,LEC,5.0,0 days 00:01:24.603000,2.0,1.0,MEDIUM,5.0,NaT,NaT,1,0 days 01:02:58.919000


In [6]:
(laps[laps["Driver"]=="NOR"]).head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
106,0 days 00:57:19.723000,NOR,4,0 days 00:01:28.971000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:31.051000,...,True,McLaren,0 days 00:55:50.494000,2024-09-01 13:03:34.413,1,3.0,False,,False,False
107,0 days 00:58:45.181000,NOR,4,0 days 00:01:25.458000,2.0,1.0,NaT,NaT,0 days 00:00:27.676000,0 days 00:00:29.408000,...,True,McLaren,0 days 00:57:19.723000,2024-09-01 13:05:03.642,1,3.0,False,,False,True
108,0 days 01:00:10.291000,NOR,4,0 days 00:01:25.110000,3.0,1.0,NaT,NaT,0 days 00:00:27.513000,0 days 00:00:29.381000,...,True,McLaren,0 days 00:58:45.181000,2024-09-01 13:06:29.100,1,3.0,False,,False,True
109,0 days 01:01:35.459000,NOR,4,0 days 00:01:25.168000,4.0,1.0,NaT,NaT,0 days 00:00:27.638000,0 days 00:00:29.468000,...,True,McLaren,0 days 01:00:10.291000,2024-09-01 13:07:54.210,1,3.0,False,,False,True
110,0 days 01:03:00.133000,NOR,4,0 days 00:01:24.674000,5.0,1.0,NaT,NaT,0 days 00:00:27.524000,0 days 00:00:29.143000,...,True,McLaren,0 days 01:01:35.459000,2024-09-01 13:09:19.378,1,3.0,False,,False,True


In [7]:
Nor = laps[laps['Driver']=="NOR"]
pit_in= Nor["PitInTime"].dropna()
pit_in.head()

119   0 days 01:15:43.308000
137   0 days 01:41:15.813000
Name: PitInTime, dtype: timedelta64[ns]

In [9]:
lec = laps.pick_drivers("LEC")

pit_in = lec.loc[lec["PitInTime"].notna(), ["LapNumber", "PitInTime", "Compound", "Stint"]]

# PitOutTime is recorded on the lap after the in-lap
pit_out = lec.loc[lec["PitOutTime"].notna(), ["LapNumber", "PitOutTime"]]
pit_out["LapNumber"] -= 1

stops = pit_in.merge(pit_out, on="LapNumber")
stops["PitLaneTime"] = stops["PitOutTime"] - stops["PitInTime"]
stops

,LapNumber,PitInTime,Compound,Stint,PitOutTime,PitLaneTime
0,15.0,0 days 01:17:08.772000,MEDIUM,1.0,0 days 01:17:33.191000,0 days 00:00:24.419000


In [10]:
laps["TrackStatus"].unique()

array(['1'], dtype=object)

In [11]:
session.race_control_messages

,Time,Category,Message,Status,Flag,Scope,Sector,RacingNumber,Lap
0,2024-09-01 12:08:03,Other,BLUE HEAD PADDING MATERIAL MUST BE USED,None,None,None,None,None,1
1,2024-09-01 12:20:01,Flag,GREEN LIGHT - PIT EXIT OPEN,None,GREEN,Track,None,None,1
2,2024-09-01 12:30:01,Other,PIT EXIT CLOSED,None,None,None,None,None,1
3,2024-09-01 12:45:09,Other,RISK OF RAIN FOR F1 RACE IS 40 %,None,None,None,None,None,1
4,2024-09-01 12:57:04,Drs,DRS DISABLED,DISABLED,None,None,None,None,1
5,2024-09-01 13:03:34,Flag,GREEN LIGHT - PIT EXIT OPEN,None,GREEN,Track,None,None,1
6,2024-09-01 13:05:13,Drs,DRS ENABLED,ENABLED,None,None,None,None,2
7,2024-09-01 13:07:52,Other,TURN 8 INCIDENT INVOLVING CARS 3 (RIC) AND 27 ...,None,None,None,None,None,3
8,2024-09-01 13:11:21,Other,TURN 1 INCIDENT INVOLVING CARS 27 (HUL) AND 22...,None,None,None,None,None,6
9,2024-09-01 13:12:02,Other,CAR 22 (TSU) TIME 1:32.241 DELETED - TRACK LIM...,None,None,None,None,None,6


In [19]:
lec = laps.pick_drivers("LEC")
pia = laps.pick_drivers("PIA")

lec_l30 = lec[lec["LapNumber"] ==30]
pia_l30 = pia[pia["LapNumber"] ==30]

gap = lec_l30["Time"].iloc[0] - pia_l30["Time"].iloc[0]
print(gap)

0 days 00:00:04.987000
